# 🎓 Scraper de Programas Académicos — Universidad de La Sabana

Extrae todos los programas del catálogo de [www.unisabana.edu.co/programas](https://www.unisabana.edu.co/programas) y genera:
- `unisabana_programas.csv` — CSV UTF-8 con BOM compatible con Excel
- `unisabana_programas.md` — Markdown agrupado por tipo
- `errores.log` — registro de URLs que fallaron

**Bloques:**
1. Instalación y configuración
2. Funciones auxiliares (con tests)
3. Paso 1 — Recolectar URLs del listado
4. Paso 2 — Enriquecer cada programa (detalle individual)
5. Guardar CSV y Markdown
6. Validaciones finales

---
### 🔧 Correcciones aplicadas vs versión anterior
| Campo | Problema original | Corrección |
|---|---|---|
| Contenedor principal | Buscaba `<main>` — no existe | Usa `<article>` |
| Facultad | Regex no lo extraía | Patrón `TIPO + nombre siguiente` |
| SNIES | Capturaba texto extra | Recorte al primer punto |
| Título otorgado | Regex fallaba por acentos | Usa búsqueda literal en texto |
| Costo COP | `$X.XXX.XXX` fallaba | Regex mejorado |
| Costo inscripción | Patrón `pesos` no contemplado | Agregado |
| Acreditación nacional | Capturaba OG description | Busca `Alta Calidad mediante Resolución` |
| Encoding | Caracteres `ó/í/é` se corrompían | `response.encoding = 'utf-8'` explícito |

---
## Bloque 0 — Instalación de dependencias

In [16]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4', 'lxml', '--quiet'])

CompletedProcess(args=['c:\\Users\\juansoag\\AppData\\Local\\miniconda3\\envs\\agrodata_env\\python.exe', '-m', 'pip', 'install', 'requests', 'beautifulsoup4', 'lxml', '--quiet'], returncode=0)

---
## Bloque 1 — Imports y configuración global

In [17]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import random
import re
import os
import pandas as pd
from collections import Counter
from datetime import datetime
from pathlib import Path

# ── Configuración ──────────────────────────────────────────────────────────────
BASE_URL        = 'https://www.unisabana.edu.co/programas'
ITEMS_PER_PAGE  = 100
DELAY_MIN       = 1.0
DELAY_MAX       = 2.0
OUTPUT_DIR      = Path('.')
CSV_FILE        = OUTPUT_DIR / 'unisabana_programas.csv'
MD_FILE         = OUTPUT_DIR / 'unisabana_programas.md'
LOG_FILE        = OUTPUT_DIR / 'errores.log'

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'es-CO,es;q=0.9,en;q=0.8',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

CSV_FIELDS = [
    'nombre', 'tipo', 'subtipo', 'facultad', 'modalidad',
    'semestres', 'creditos',
    'costo_semestral_cop', 'costo_semestral_usd', 'costo_inscripcion_cop',
    'codigo_snies', 'titulo_otorgado',
    'acreditacion_nacional', 'acreditacion_internacional',
    'descripcion', 'keywords', 'url'
]

print('✅ Configuración lista')
print(f'   → URL base:         {BASE_URL}')
print(f'   → Items por página: {ITEMS_PER_PAGE}')
print(f'   → Salida CSV:       {CSV_FILE.resolve()}')
print(f'   → Salida Markdown:  {MD_FILE.resolve()}')

✅ Configuración lista
   → URL base:         https://www.unisabana.edu.co/programas
   → Items por página: 100
   → Salida CSV:       C:\Users\juansoag\Downloads\Github\Newsletter\Scrappers\unisabana_programas.csv
   → Salida Markdown:  C:\Users\juansoag\Downloads\Github\Newsletter\Scrappers\unisabana_programas.md


---
## Bloque 2 — Funciones auxiliares

In [18]:
def log_error(url: str, reason: str):
    """Registra errores en errores.log sin interrumpir el scraper."""
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        f.write(f'[{ts}] ERROR | {url} | {reason}\n')


def clean_text(text: str) -> str:
    """Normaliza espacios y saltos de línea en texto extraído del HTML."""
    if not text:
        return ''
    return ' '.join(str(text).split())


def extract_number(text: str) -> int | None:
    """
    Extrae el primer número entero de un fragmento de texto.
    Elimina separadores de miles (puntos y comas en formato colombiano).
    Ejemplos:
      '$20.321.000 COP'    → 20321000
      'USD 5,110 aprox.'   → 5110
      '130.000 pesos'      → 130000
      'Sin costo'          → None
    """
    if not text:
        return None
    cleaned = re.sub(r'[^\d]', '', text)
    return int(cleaned) if cleaned else None


def safe_get(session: requests.Session, url: str, timeout: int = 25) -> BeautifulSoup | None:
    """
    GET seguro con manejo de errores y encoding explícito.
    CORRECCIÓN: response.encoding = 'utf-8' evita que los acentos se corrompan.
    Devuelve soup o None si hay error.
    """
    try:
        response = session.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status()
        response.encoding = 'utf-8'  # ← forzar UTF-8 para caracteres con acento
        return BeautifulSoup(response.text, 'lxml')
    except requests.exceptions.RequestException as e:
        log_error(url, str(e))
        return None


def infer_subtipo(nombre: str, tipo: str) -> str:
    """
    Infiere el subtipo a partir del nombre del programa.
    """
    n = nombre.lower()
    if 'especialización' in n or 'especializacion' in n: return 'Especialización'
    if 'maestría' in n or 'maestria' in n:               return 'Maestría'
    if 'doctorado' in n:                                  return 'Doctorado'
    if 'diplomado' in n:                                  return 'Diplomado'
    if 'seminario' in n:                                  return 'Seminario'
    if 'certificado' in n:                                return 'Certificado'
    if 'curso' in n:                                      return 'Curso'
    return ''


print('✅ Funciones auxiliares definidas:')
print('   → log_error(), clean_text(), extract_number(), safe_get(), infer_subtipo()')

# ── Tests ──────────────────────────────────────────────────────────────────────
print('\n🧪 Tests de extract_number():')
tests = [
    ('$20.321.000 COP', 20321000),
    ('USD 5,110 aprox.', 5110),
    ('130.000 pesos',    130000),
    ('Sin costo',        None),
    ('34.419.000',       34419000),
]
all_ok = True
for inp, expected in tests:
    result = extract_number(inp)
    status = '✅' if result == expected else '❌'
    if result != expected: all_ok = False
    print(f'   {status}  extract_number({inp!r:30}) = {str(result):<12} (esperado: {expected})')
print(f'\n   Resultado: {"TODOS OK ✅" if all_ok else "HAY FALLOS ❌"}')

✅ Funciones auxiliares definidas:
   → log_error(), clean_text(), extract_number(), safe_get(), infer_subtipo()

🧪 Tests de extract_number():
   ✅  extract_number('$20.321.000 COP'             ) = 20321000     (esperado: 20321000)
   ✅  extract_number('USD 5,110 aprox.'            ) = 5110         (esperado: 5110)
   ✅  extract_number('130.000 pesos'               ) = 130000       (esperado: 130000)
   ✅  extract_number('Sin costo'                   ) = None         (esperado: None)
   ✅  extract_number('34.419.000'                  ) = 34419000     (esperado: 34419000)

   Resultado: TODOS OK ✅


---
## Bloque 3 — Paso 1: Recolectar URLs del listado

Itera sobre las páginas del catálogo con `items_per_page=100` (reduce ~33 páginas a ~4).
De cada tarjeta extrae: nombre, tipo, modalidad, semestres, créditos y URL individual.

In [19]:
def parse_card(card) -> dict | None:
    """
    Parsea una tarjeta de programa del listado.
    Devuelve un dict con los campos básicos, o None si no es válida.
    """
    link_tag = card.find('a', href=True)
    if not link_tag:
        return None
    url = link_tag['href'].strip()
    if not url or not url.startswith('http'):
        if url.startswith('/'):
            url = 'https://www.unisabana.edu.co' + url
        else:
            return None

    # Nombre: preferir <h3> sobre el texto del link
    h3 = card.find('h3')
    nombre = clean_text(h3.get_text()) if h3 else clean_text(link_tag.get_text())
    if not nombre:
        return None

    text = clean_text(card.get_text(' '))
    text_lower = text.lower()

    # Tipo — inferir del texto visible en la tarjeta
    tipo = ''
    if   'educación continua' in text_lower or 'educacion continua' in text_lower:
        tipo = 'Educación Continua'
    elif 'pregrado' in text_lower:
        tipo = 'Pregrado'
    elif 'posgrado' in text_lower:
        tipo = 'Posgrado'
    elif 'técnico' in text_lower or 'tecnico' in text_lower:
        tipo = 'Técnico Laboral'
    elif 'minor' in text_lower:
        tipo = 'Minor'

    # Si no detectó tipo en texto, inferirlo de la URL
    if not tipo:
        if '/pregrados/' in url:           tipo = 'Pregrado'
        elif '/posgrados/' in url:         tipo = 'Posgrado'
        elif '/educacion-continua/' in url: tipo = 'Educación Continua'
        elif '/tecnico' in url:            tipo = 'Técnico Laboral'
        elif '/minor' in url:              tipo = 'Minor'

    # Modalidad
    modalidad = ''
    for m in ['Presencial-Virtual', 'Hyflex', 'Híbrido', 'Virtual', 'Presencial']:
        if m.lower().replace('í','i') in text_lower.replace('í','i'):
            modalidad = m
            break

    # Semestres
    semestres = None
    ms = re.search(r'(\d+)\s*semestre', text, re.IGNORECASE)
    if ms:
        semestres = int(ms.group(1))

    # Créditos
    creditos = None
    mc = re.search(r'(\d+)\s*cr[eé]dito', text, re.IGNORECASE)
    if mc:
        creditos = int(mc.group(1))

    return {
        'nombre':                   nombre,
        'tipo':                     tipo,
        'subtipo':                  infer_subtipo(nombre, tipo),
        'modalidad':                modalidad,
        'semestres':                semestres,
        'creditos':                 creditos,
        'url':                      url,
        # Campos que se completan en el Paso 2
        'facultad':                 '',
        'costo_semestral_cop':      None,
        'costo_semestral_usd':      None,
        'costo_inscripcion_cop':    None,
        'codigo_snies':             '',
        'titulo_otorgado':          '',
        'acreditacion_nacional':    '',
        'acreditacion_internacional': '',
        'descripcion':              '',
        'keywords':                 '',
    }


def get_program_urls(session: requests.Session, items_per_page: int = 100) -> list[dict]:
    """
    Itera sobre todas las páginas del listado y devuelve lista de dicts
    con datos básicos de cada programa (sin duplicados).
    """
    programs  = []
    seen_urls = set()
    page      = 0

    while True:
        url = f'{BASE_URL}?items_per_page={items_per_page}&page=,{page}'
        print(f'  📄 Página {page}: {url}')

        soup = safe_get(session, url)
        if soup is None:
            print(f'     ⚠️  No se pudo obtener la página {page}. Deteniendo.')
            break

        # Buscar tarjetas: <article>, <li> o <div> que contengan un <h3>
        # y un enlace a /programas/
        cards = []
        for tag_name in ['article', 'li', 'div']:
            for c in soup.find_all(tag_name):
                link = c.find('a', href=re.compile(r'/programas/'))
                h3   = c.find('h3')
                if link and h3:
                    cards.append(c)
            if cards:
                break

        if not cards:
            print(f'     ℹ️  Sin tarjetas en página {page}. Fin del catálogo.')
            break

        new_in_page = 0
        for card in cards:
            data = parse_card(card)
            if data and data['url'] not in seen_urls:
                seen_urls.add(data['url'])
                programs.append(data)
                new_in_page += 1

        print(f'     ✅ {new_in_page} programas nuevos (acumulado: {len(programs)})')

        if new_in_page == 0:
            print('     ℹ️  Sin programas nuevos. Fin del catálogo.')
            break

        page += 1
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    return programs


print('✅ Funciones de listado definidas: parse_card(), get_program_urls()')

✅ Funciones de listado definidas: parse_card(), get_program_urls()


In [20]:
# ── Ejecutar Paso 1 ───────────────────────────────────────────────────────────
session = requests.Session()
session.headers.update(HEADERS)

print('🚀 Iniciando recolección de URLs del listado...\n')
programs = get_program_urls(session, items_per_page=ITEMS_PER_PAGE)

print(f'\n📊 RESUMEN PASO 1')
print(f'   Total programas encontrados: {len(programs)}')
tipos_count = Counter(p['tipo'] or 'Sin tipo' for p in programs)
for tipo, count in sorted(tipos_count.items()):
    print(f'   - {tipo}: {count}')

🚀 Iniciando recolección de URLs del listado...

  📄 Página 0: https://www.unisabana.edu.co/programas?items_per_page=100&page=,0
     ✅ 101 programas nuevos (acumulado: 101)
  📄 Página 1: https://www.unisabana.edu.co/programas?items_per_page=100&page=,1
     ✅ 99 programas nuevos (acumulado: 200)
  📄 Página 2: https://www.unisabana.edu.co/programas?items_per_page=100&page=,2
     ✅ 96 programas nuevos (acumulado: 296)
  📄 Página 3: https://www.unisabana.edu.co/programas?items_per_page=100&page=,3
     ✅ 90 programas nuevos (acumulado: 386)
  📄 Página 4: https://www.unisabana.edu.co/programas?items_per_page=100&page=,4
     ℹ️  Sin tarjetas en página 4. Fin del catálogo.

📊 RESUMEN PASO 1
   Total programas encontrados: 386
   - Educación Continua: 193
   - Minor: 19
   - Posgrado: 140
   - Pregrado: 29
   - Técnico Laboral: 5


In [21]:
# ── Verificación Paso 1: head() ───────────────────────────────────────────────
df_paso1 = pd.DataFrame(programs)
print(f'Shape: {df_paso1.shape}')
print('\n── Primeros 10 programas ────────────────────────────────────────────────')
display(df_paso1[['nombre','tipo','subtipo','modalidad','semestres','creditos','url']].head(10))

print('\n── Valores vacíos en campos básicos ─────────────────────────────────────')
for col in ['tipo', 'modalidad', 'semestres', 'creditos']:
    vacíos = df_paso1[col].isna().sum() + (df_paso1[col] == '').sum()
    print(f'   {col:<15}: {vacíos} vacíos')

Shape: (386, 17)

── Primeros 10 programas ────────────────────────────────────────────────


,nombre,tipo,subtipo,modalidad,semestres,creditos,url
0,Escuela Internacional de Ciencias Económicas y...,Educación Continua,,Hyflex,9.0,162.0,https://www.unisabana.edu.co/
1,Administración & Servicio,Pregrado,,Presencial,9.0,162.0,https://www.unisabana.edu.co/programas/pregrad...
2,Administración de Mercadeo y Logística Interna...,Pregrado,,Presencial,9.0,162.0,https://www.unisabana.edu.co/programas/pregrad...
3,Gastronomía,Pregrado,,Presencial,9.0,162.0,https://www.unisabana.edu.co/programas/pregrad...
4,Economía y Finanzas Internacionales,Pregrado,,Presencial,9.0,162.0,https://www.unisabana.edu.co/programas/pregrad...
5,Administración de Empresas,Pregrado,,Presencial,9.0,162.0,https://www.unisabana.edu.co/programas/pregrad...
6,Administración de Negocios Internacionales,Pregrado,,Presencial,10.0,180.0,https://www.unisabana.edu.co/programas/pregrad...
7,Economía y Finanzas Internacionales Virtual,Pregrado,,Virtual,8.0,153.0,https://www.unisabana.edu.co/programas/pregrad...
8,Doctorado en Administración de Organizaciones,Posgrado,Doctorado,Presencial,NaN,90.0,https://www.unisabana.edu.co/programas/posgrad...
9,Maestría en Gerencia de Negocios de la Hospita...,Posgrado,Maestría,Virtual,3.0,49.0,https://www.unisabana.edu.co/programas/posgrad...



── Valores vacíos en campos básicos ─────────────────────────────────────
   tipo           : 0 vacíos
   modalidad      : 0 vacíos
   semestres      : 219 vacíos
   creditos       : 205 vacíos


---
## Bloque 4 — Paso 2: Enriquecer cada programa (detalle individual)

Para cada URL extraída en el Paso 1:
- **`<head>`**: `og:title`, `og:description`, `keywords`  
- **`<article>`**: SNIES, costos, título otorgado, acreditaciones, facultad
  - ⚠️ El sitio no usa `<main>` — el contenedor real es `<article>`

Se guarda **incrementalmente** en CSV (fila a fila) para no perder progreso si se interrumpe.

In [22]:
def extract_from_head(soup: BeautifulSoup) -> dict:
    """Extrae metadata del <head>: og:title, og:description, keywords, og:url."""
    data = {'nombre_og': '', 'descripcion': '', 'keywords': '', 'url_canonical': ''}

    og_title = soup.find('meta', property='og:title')
    if og_title:
        data['nombre_og'] = clean_text(og_title.get('content', ''))

    og_desc = soup.find('meta', property='og:description')
    if og_desc:
        data['descripcion'] = clean_text(og_desc.get('content', ''))

    og_url = soup.find('meta', property='og:url')
    if og_url:
        data['url_canonical'] = og_url.get('content', '').strip()

    kw_tag = soup.find('meta', attrs={'name': 'keywords'})
    if kw_tag:
        data['keywords'] = clean_text(kw_tag.get('content', ''))

    return data


def extract_from_article(soup: BeautifulSoup) -> dict:
    """
    CORRECCIÓN PRINCIPAL: el sitio de Unisabana NO usa <main> como contenedor.
    El contenido del programa está dentro de <article>.
    Se eliminan <select>, <option>, <script>, <style>, <nav>, <footer> del árbol
    antes de extraer texto para evitar el 'ruido' del dropdown de colegios.
    """
    data = {
        'facultad':                   '',
        'semestres':                  None,
        'creditos':                   None,
        'costo_semestral_cop':        None,
        'costo_semestral_usd':        None,
        'costo_inscripcion_cop':      None,
        'codigo_snies':               '',
        'titulo_otorgado':            '',
        'acreditacion_nacional':      '',
        'acreditacion_internacional': '',
    }

    # CORRECCIÓN: buscar <article>, no <main>
    article = soup.find('article')
    if article is None:
        # Fallback: div#content o div con clase 'field--type-text'
        article = soup.find(id='content') or soup.find(class_=re.compile('field-items'))
    if article is None:
        return data

    # Eliminar ruido antes de parsear
    for tag in article.find_all(['select', 'option', 'script', 'style', 'nav', 'footer']):
        tag.decompose()

    text = clean_text(article.get_text(' '))

    # ── SNIES ─────────────────────────────────────────────────────────────────
    # HTML real: <strong>Código SNIES:</strong><p>104355. Esta carrera...</p>
    # CORRECCIÓN: recortar al primer punto para no incluir texto extra
    m = re.search(r'SNIES[:\s]*(\d+)', text)
    if m:
        data['codigo_snies'] = m.group(1).strip()

    # ── Título otorgado ───────────────────────────────────────────────────────
    # HTML real: "Título que otorga: Profesional en Administración & Servicio."
    # CORRECCIÓN: usar \w+ para ignorar acentos en la regex, capturar hasta el punto
    m = re.search(r'[Tt][ií]tulo\s+que\s+otorga\s*[:\-]?\s*([^.\n]{5,120})', text)
    if not m:
        m = re.search(r'[Tt][ií]tulo\s+que\s+otorga\s*[:\-]?\s*([^\n]{5,120})', text)
    if m:
        data['titulo_otorgado'] = clean_text(m.group(1))

    # ── Costo semestral COP ────────────────────────────────────────────────────
    # HTML real: "matr\u00edcula por semestre para 2026: $20.321.000 COP"
    # o en el hero: "20.321.000 COP / USD 5,110"
    # CORRECCIÓN: búsqueda directa del patrón número+COP
    m = re.search(r'([\d\.]{7,12})\s*COP', text)  # mínimo 7 dígitos (con puntos)
    if m:
        raw = m.group(1).replace('.', '')
        if raw.isdigit() and len(raw) >= 5:
            data['costo_semestral_cop'] = int(raw)

    # ── Costo semestral USD ───────────────────────────────────────────────────
    # HTML real: "USD 5,110 aprox." o "USD 9.060"
    m = re.search(r'USD\s*([\d\.,]+)', text)
    if m:
        raw = re.sub(r'[^\d]', '', m.group(1))
        if raw:
            data['costo_semestral_usd'] = int(raw)

    # ── Costo inscripción COP ─────────────────────────────────────────────────
    # HTML real: "El valor de la inscripción: 130.000 pesos"
    # CORRECCIÓN: agregar patrón con 'pesos' además de '$'
    m_ins = re.search(
        r'inscripci[oó]n[:\s]+\$?\s*([\d\.]+)\s*(?:pesos|COP)?',
        text, re.IGNORECASE
    )
    if m_ins:
        raw = m_ins.group(1).replace('.', '')
        if raw.isdigit():
            data['costo_inscripcion_cop'] = int(raw)

    # ── Semestres ─────────────────────────────────────────────────────────────
    m = re.search(r'(\d+)\s*semestre', text, re.IGNORECASE)
    if m:
        data['semestres'] = int(m.group(1))

    # ── Créditos ──────────────────────────────────────────────────────────────
    m = re.search(r'(\d+)\s*cr[eé]dito', text, re.IGNORECASE)
    if m:
        data['creditos'] = int(m.group(1))

    # ── Acreditación nacional ─────────────────────────────────────────────────
    # HTML real: "Cuarta renovación de Acreditación de Alta Calidad mediante
    #             Resolución No. 016907 del 20 de septiembre de 2023. Vigencia por 8 años."
    # CORRECCIÓN: buscar la Resolución específica, no el texto genérico de OG
    m = re.search(
        r'((?:renovaci[oó]n\s+de\s+)?Acreditaci[oó]n\s+de\s+Alta\s+Calidad[^.]{0,200}?Vigencia[^.]{0,60})',
        text, re.IGNORECASE
    )
    if not m:
        # Fallback: buscar sólo "Alta Calidad" + Resolución
        m = re.search(
            r'Alta\s+Calidad[^.]{0,80}?Resoluci[oó]n\s+No\.?\s*([\d]+)[^.]{0,100}',
            text, re.IGNORECASE
        )
    if not m:
        # Último fallback: Res. + número
        m = re.search(
            r'Acreditaci[oó]n\s+mediante\s+Resoluci[oó]n[^.]{0,150}',
            text, re.IGNORECASE
        )
    if m:
        data['acreditacion_nacional'] = clean_text(m.group(0))[:250]

    # ── Acreditación internacional ────────────────────────────────────────────
    for kw in ['EFMD', 'AACSB', 'AMBA', 'EQUIS', 'PMI', 'ACBSP', 'ABET', 'ACCREDITATION']:
        m = re.search(rf'{kw}[^.\n]{{0,150}}', text, re.IGNORECASE)
        if m:
            data['acreditacion_internacional'] = clean_text(m.group(0))[:200]
            break

    # ── Facultad ──────────────────────────────────────────────────────────────
    # HTML real: "PREGRADOS Escuela Internacional de Ciencias Económicas y Administrativas
    #             Administración & Servicio ..."
    # CORRECCIÓN: capturar el nombre que aparece inmediatamente después de PREGRADO(S)/POSGRADO(S)
    for tipo_label in ['PREGRADOS', 'POSGRADOS', 'EDUCACI.N CONTINUA']:
        m = re.search(
            rf'{tipo_label}\s+([^\n]{{10,100}}?)(?:\s+(?:Administraci|Comunicaci|Ingenier|Medicin|Psicolog|Filosofí|Biomedic|Enfermeri|Nutri))',
            text, re.IGNORECASE
        )
        if m:
            data['facultad'] = clean_text(m.group(1))
            break

    # Fallback: búsqueda directa de "Facultad de" o "Escuela Internacional"
    if not data['facultad']:
        m = re.search(
            r'((?:Facultad|Escuela Internacional|Instituto)\s+(?:de\s+|para\s+)?[A-ZÁÉÍÓÚ][^\n\.]{5,80})',
            text
        )
        if m:
            candidate = clean_text(m.group(1))
            # Descartar si parece parte de una lista de navegación
            if len(candidate.split()) < 15:
                data['facultad'] = candidate

    return data


def scrape_program_detail(url: str, session: requests.Session) -> dict:
    """Para una URL individual, combina datos del <head> y del <article>."""
    soup = safe_get(session, url)
    if soup is None:
        return {}

    head_data    = extract_from_head(soup)
    article_data = extract_from_article(soup)

    return {
        'nombre_og':               head_data['nombre_og'],
        'descripcion':             head_data['descripcion'],
        'keywords':                head_data['keywords'],
        'url':                     head_data.get('url_canonical') or url,
        'facultad':                article_data['facultad'],
        'semestres':               article_data['semestres'],
        'creditos':                article_data['creditos'],
        'costo_semestral_cop':     article_data['costo_semestral_cop'],
        'costo_semestral_usd':     article_data['costo_semestral_usd'],
        'costo_inscripcion_cop':   article_data['costo_inscripcion_cop'],
        'codigo_snies':            article_data['codigo_snies'],
        'titulo_otorgado':         article_data['titulo_otorgado'],
        'acreditacion_nacional':   article_data['acreditacion_nacional'],
        'acreditacion_internacional': article_data['acreditacion_internacional'],
    }


print('✅ Funciones de detalle definidas:')
print('   → extract_from_head(), extract_from_article(), scrape_program_detail()')

✅ Funciones de detalle definidas:
   → extract_from_head(), extract_from_article(), scrape_program_detail()


In [23]:
# ── Test en 3 programas conocidos antes de correr el scraper completo ─────────
TEST_URLS = [
    ('Administración & Servicio (pregrado con costo y acreditación)',
     'https://www.unisabana.edu.co/programas/pregrados/administracion-y-servicio'),
    ('Maestría en Gerencia de Negocios de la Hospitalidad (sin costo)',
     'https://www.unisabana.edu.co/programas/posgrados/maestria-en-gerencia-de-negocios-de-la-hospitalidad'),
    ('Medicina (pregrado con costo pero sin USD separado)',
     'https://www.unisabana.edu.co/programas/pregrados/medicina'),
]

print('🧪 TEST DE scrape_program_detail() en 3 URLs representativas\n')

for desc, url in TEST_URLS:
    print(f'\n{"─"*70}')
    print(f'🔗 {desc}')
    print(f'   {url}')
    detail = scrape_program_detail(url, session)
    campos_criticos = ['nombre_og','facultad','codigo_snies','titulo_otorgado',
                       'costo_semestral_cop','costo_semestral_usd','costo_inscripcion_cop',
                       'acreditacion_nacional','acreditacion_internacional']
    for k in campos_criticos:
        v = detail.get(k)
        estado = '✅' if v else '⬜'
        val_str = str(v)[:80] if v else '(vacío)'
        print(f'   {estado} {k:<35}: {val_str}')
    time.sleep(1.5)

print(f'\n{"─"*70}')
print('\n⚠️  Verificar:')
print('   - SNIES debería ser solo el número (ej: 104355)')
print('   - Título: string limpio (ej: Profesional en Administración & Servicio)')
print('   - Acreditación nacional: texto con Resolución y vigencia')
print('   - Facultad: nombre de escuela/facultad (ej: Escuela Internacional...)')

🧪 TEST DE scrape_program_detail() en 3 URLs representativas


──────────────────────────────────────────────────────────────────────
🔗 Administración & Servicio (pregrado con costo y acreditación)
   https://www.unisabana.edu.co/programas/pregrados/administracion-y-servicio
   ✅ nombre_og                          : Administración & Servicio
   ✅ facultad                           : Escuela Internacional de Ciencias Económicas y Administrativas
   ✅ codigo_snies                       : 104355
   ✅ titulo_otorgado                    : Profesional en Administración & Servicio
   ✅ costo_semestral_cop                : 20321000
   ✅ costo_semestral_usd                : 5110
   ✅ costo_inscripcion_cop              : 130000
   ✅ acreditacion_nacional              : Alta Calidad mediante Resolución No. 016907 del 20 de septiembre de 2023
   ✅ acreditacion_internacional         : EFMD

──────────────────────────────────────────────────────────────────────
🔗 Maestría en Gerencia de Negocios de l

In [24]:
# ── Inicializar CSV incremental ───────────────────────────────────────────────
def init_csv(filepath: Path, fields: list):
    """Crea el CSV con la fila de encabezados (UTF-8 con BOM para Excel)."""
    with open(filepath, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fields, quoting=csv.QUOTE_ALL)
        writer.writeheader()


def append_to_csv(filepath: Path, fields: list, row: dict):
    """Agrega una fila al CSV. Los valores None se convierten en string vacío."""
    clean_row = {k: ('' if row.get(k) is None else row.get(k, '')) for k in fields}
    with open(filepath, 'a', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fields, quoting=csv.QUOTE_ALL)
        writer.writerow(clean_row)


init_csv(CSV_FILE, CSV_FIELDS)
print(f'✅ CSV inicializado (vacío, sólo headers): {CSV_FILE.resolve()}')

✅ CSV inicializado (vacío, sólo headers): C:\Users\juansoag\Downloads\Github\Newsletter\Scrappers\unisabana_programas.csv


In [25]:
# ── Ejecutar Paso 2: Enriquecer todos los programas ───────────────────────────
print(f'🚀 Iniciando enriquecimiento de {len(programs)} programas...')
print('   Guardando incrementalmente en CSV — puedes interrumpir y retomar.\n')

errores   = []
t_inicio  = datetime.now()

for i, program in enumerate(programs):
    url = program['url']

    # Progreso cada 10 programas
    if i % 10 == 0:
        elapsed = (datetime.now() - t_inicio).seconds
        print(f'   [{i+1:>3}/{len(programs)}] ⏱ {elapsed}s — {program["nombre"][:55]}...')

    detail = scrape_program_detail(url, session)

    if not detail:
        errores.append(url)
        append_to_csv(CSV_FILE, CSV_FIELDS, program)
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        continue

    # Actualizar el programa con datos del detalle
    # ── Semestres y créditos: los del <article> son más fiables
    for field in ['semestres', 'creditos']:
        if detail.get(field) is not None:
            program[field] = detail[field]

    # ── Campos que SÓLO vienen del detalle
    for field in ['facultad', 'costo_semestral_cop', 'costo_semestral_usd',
                  'costo_inscripcion_cop', 'codigo_snies', 'titulo_otorgado',
                  'acreditacion_nacional', 'acreditacion_internacional',
                  'descripcion', 'keywords', 'url']:
        if detail.get(field):  # sólo sobreescribir si el detalle trae algo
            program[field] = detail[field]

    # ── Nombre: preferir og:title si es más completo o igual
    if detail.get('nombre_og'):
        # Usar og:title si el nombre del listado era la escuela/facultad (error de parseo)
        if len(detail['nombre_og']) <= len(program.get('nombre', '')) + 10:
            program['nombre'] = detail['nombre_og']

    append_to_csv(CSV_FILE, CSV_FIELDS, program)
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

elapsed_total = (datetime.now() - t_inicio).seconds
print(f'\n✅ Paso 2 completado en {elapsed_total}s')
print(f'   Programas procesados: {len(programs)}')
print(f'   Errores:              {len(errores)}')
if errores:
    for e in errores[:10]:
        print(f'     ❌ {e}')

🚀 Iniciando enriquecimiento de 386 programas...
   Guardando incrementalmente en CSV — puedes interrumpir y retomar.

   [  1/386] ⏱ 0s — Escuela Internacional de Ciencias Económicas y Administ...
   [ 11/386] ⏱ 52s — Maestría en Gerencia de Inversión...
   [ 21/386] ⏱ 87s — Minor en Trading...
   [ 31/386] ⏱ 137s — Maestría en Periodismo y Comunicación Digital...
   [ 41/386] ⏱ 180s — Diplomado en Creación de Contenidos Deportivos...
   [ 51/386] ⏱ 216s — Minor en Creación de Videojuegos...
   [ 61/386] ⏱ 260s — Maestría en Derecho Internacional...
   [ 71/386] ⏱ 308s — Doctorado en Derecho...
   [ 81/386] ⏱ 350s — Diplomado en Gobierno Corporativo en la práctica: Poder...
   [ 91/386] ⏱ 397s — Diplomado en ambiente, desarrollo territorial y adaptac...
   [101/386] ⏱ 430s — Unisabana College...
   [111/386] ⏱ 479s — Maestría en Pedagogía...
   [121/386] ⏱ 528s — Diplomado en Convivencia Escolar y Educación Emocional...
   [131/386] ⏱ 582s — Diplomado en Ventilación Mecánica...
   [141

In [26]:
# ── Verificación: leer el CSV generado ───────────────────────────────────────
df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
print(f'📋 CSV generado — Shape: {df.shape}')

print('\n── Primeras 5 filas (campos críticos) ───────────────────────────────────')
display(df[['nombre','tipo','facultad','codigo_snies','titulo_otorgado',
            'costo_semestral_cop','costo_semestral_usd','costo_inscripcion_cop']].head())

print('\n── Últimas 5 filas ──────────────────────────────────────────────────────')
display(df[['nombre','tipo','facultad','codigo_snies','titulo_otorgado']].tail())

print('\n── Valores no vacíos por columna ────────────────────────────────────────')
total = len(df)
for col in CSV_FIELDS:
    n = df[col].notna().sum() - (df[col] == '').sum()
    pct = n / total * 100
    bar = '█' * int(pct / 5)
    print(f'  {col:<30} {n:>4}/{total}  {pct:>5.1f}%  {bar}')

📋 CSV generado — Shape: (386, 17)

── Primeras 5 filas (campos críticos) ───────────────────────────────────


,nombre,tipo,facultad,codigo_snies,titulo_otorgado,costo_semestral_cop,costo_semestral_usd,costo_inscripcion_cop
0,Universidad de la Sabana | Universidad de la S...,Educación Continua,Escuela Internacional de Ciencias Económicas y...,NaN,NaN,NaN,NaN,NaN
1,Administración & Servicio,Pregrado,Escuela Internacional de Ciencias Económicas y...,104355.0,Profesional en Administración & Servicio,20321000.0,5110.0,130000.0
2,Administración de Mercadeo y Logística Interna...,Pregrado,Escuela Internacional de Ciencias Económicas y...,101896.0,Administrador de Mercadeo y Logística Internac...,20321000.0,5110.0,130000.0
3,Gastronomía,Pregrado,Facultad Continúa tu formación Pregrados Relac...,53517.0,Gastrónomo,20321000.0,5110.0,130000.0
4,Economía y Finanzas Internacionales,Pregrado,Facultad Continúa tu formación Pregrados Relac...,53516.0,Economista con énfasis en Finanzas Internacion...,20321000.0,5110.0,130000.0



── Últimas 5 filas ──────────────────────────────────────────────────────


,nombre,tipo,facultad,codigo_snies,titulo_otorgado
381,Técnico Laboral en Alimentos y Bebidas,Técnico Laboral,NaN,NaN,Certificado aptitud ocupacional Técnico Labora...
382,Técnico Laboral en Manejo de Herramientas para...,Técnico Laboral,NaN,NaN,Certificado de aptitud ocupacional de técnico ...
383,Técnico Laboral en Auxiliar en Ventas,Técnico Laboral,NaN,NaN,Certificado de aptitud ocupacional en Técnico ...
384,Técnico Laboral en Asistencia y Soporte de Tec...,Técnico Laboral,NaN,NaN,Aptitud Ocupacional en Técnico Laboral en Asis...
385,Técnico Laboral en Cuidado Integral de la Prim...,Técnico Laboral,NaN,NaN,Certificado de aptitud ocupacional en Técnico ...



── Valores no vacíos por columna ────────────────────────────────────────
  nombre                          386/386  100.0%  ████████████████████
  tipo                            386/386  100.0%  ████████████████████
  subtipo                         323/386   83.7%  ████████████████
  facultad                        315/386   81.6%  ████████████████
  modalidad                       386/386  100.0%  ████████████████████
  semestres                       174/386   45.1%  █████████
  creditos                        198/386   51.3%  ██████████
  costo_semestral_cop             233/386   60.4%  ████████████
  costo_semestral_usd              73/386   18.9%  ███
  costo_inscripcion_cop           134/386   34.7%  ██████
  codigo_snies                    156/386   40.4%  ████████
  titulo_otorgado                  84/386   21.8%  ████
  acreditacion_nacional            17/386    4.4%  
  acreditacion_internacional      143/386   37.0%  ███████
  descripcion                     384/386   99

---
## Bloque 5 — Guardar Markdown agrupado por tipo

In [27]:
def save_to_markdown(programs: list, filepath: Path):
    """Genera el archivo Markdown con los programas agrupados por tipo."""
    today = datetime.now().strftime('%Y-%m-%d')
    lines = [
        '# Programas Académicos — Universidad de La Sabana',
        f'> Extraído el {today}. Total: {len(programs)} programas.',
        '',
    ]

    tipo_order = ['Pregrado', 'Posgrado', 'Educación Continua', 'Técnico Laboral', 'Minor', '']
    grouped = {}
    for p in programs:
        t = p.get('tipo') or ''
        grouped.setdefault(t, []).append(p)

    for tipo in tipo_order:
        grupo = grouped.get(tipo, [])
        if not grupo:
            continue

        section = tipo if tipo else 'Sin clasificar'
        lines += [f'## {section} ({len(grupo)})', '']

        for p in grupo:
            nombre = p.get('nombre', 'Sin nombre')
            lines.append(f'### {nombre}')

            campos = [
                ('subtipo',                  'Subtipo',                  '{}'),
                ('facultad',                 'Facultad',                 '{}'),
                ('modalidad',               'Modalidad',                '{}'),
            ]
            for key, label, fmt in campos:
                v = p.get(key)
                if v:
                    lines.append(f'- **{label}:** {fmt.format(v)}')

            dur = []
            if p.get('semestres'): dur.append(f'{p["semestres"]} semestres')
            if p.get('creditos'):  dur.append(f'{p["creditos"]} créditos')
            if dur:
                lines.append(f'- **Duración:** {" — ".join(dur)}')

            costo = []
            if p.get('costo_semestral_cop'):
                costo.append(f'${p["costo_semestral_cop"]:,} COP'.replace(',','.'))
            if p.get('costo_semestral_usd'):
                costo.append(f'USD {p["costo_semestral_usd"]:,}')
            if costo:
                lines.append(f'- **Costo semestral:** {" / ".join(costo)}')
            if p.get('costo_inscripcion_cop'):
                lines.append(f'- **Costo inscripción:** ${p["costo_inscripcion_cop"]:,} COP'.replace(',','.'))

            if p.get('codigo_snies'):
                lines.append(f'- **Código SNIES:** {p["codigo_snies"]}')
            if p.get('titulo_otorgado'):
                lines.append(f'- **Título:** {p["titulo_otorgado"]}')
            if p.get('acreditacion_nacional'):
                lines.append(f'- **Acreditación nacional:** {p["acreditacion_nacional"]}')
            if p.get('acreditacion_internacional'):
                lines.append(f'- **Acreditación internacional:** {p["acreditacion_internacional"]}')
            if p.get('descripcion'):
                desc = p['descripcion'][:300] + ('...' if len(p['descripcion']) > 300 else '')
                lines.append(f'- **Descripción:** {desc}')
            if p.get('keywords'):
                lines.append(f'- **Keywords:** {p["keywords"]}')
            if p.get('url'):
                lines.append(f'- **URL:** {p["url"]}')
            lines.append('')

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))

    print(f'✅ Markdown guardado: {filepath.resolve()}')
    secciones = [t for t in tipo_order if t in grouped]
    for s in secciones:
        print(f'   - {s}: {len(grouped[s])} programas')


save_to_markdown(programs, MD_FILE)

print('\n── Vista previa (primeras 70 líneas) ────────────────────────────────────')
with open(MD_FILE, encoding='utf-8') as f:
    print(''.join(f.readlines()[:70]))

✅ Markdown guardado: C:\Users\juansoag\Downloads\Github\Newsletter\Scrappers\unisabana_programas.md
   - Pregrado: 29 programas
   - Posgrado: 140 programas
   - Educación Continua: 193 programas
   - Técnico Laboral: 5 programas
   - Minor: 19 programas

── Vista previa (primeras 70 líneas) ────────────────────────────────────
# Programas Académicos — Universidad de La Sabana
> Extraído el 2026-06-12. Total: 386 programas.

## Pregrado (29)

### Administración & Servicio
- **Facultad:** Escuela Internacional de Ciencias Económicas y Administrativas
- **Modalidad:** Presencial
- **Duración:** 9 semestres — 162 créditos
- **Costo semestral:** $20.321.000 COP / USD 5,110
- **Costo inscripción:** $130.000 COP
- **Código SNIES:** 104355
- **Título:** Profesional en Administración & Servicio
- **Acreditación nacional:** Alta Calidad mediante Resolución No. 016907 del 20 de septiembre de 2023
- **Acreditación internacional:** EFMD
- **Descripción:** La carrera de Administración & Servicio de

---
## Bloque 6 — Validaciones y resumen final

In [28]:
df_final = pd.read_csv(CSV_FILE, encoding='utf-8-sig')

print('=' * 65)
print(' VALIDACIONES FINALES — Scraper Universidad de La Sabana')
print('=' * 65)
print(f'\n📊 Total de programas extraídos: {len(df_final)}')

print('\n── Desglose por tipo ────────────────────────────────────────────────────')
tipo_counts = df_final['tipo'].value_counts(dropna=False)
for tipo, count in tipo_counts.items():
    label = tipo if isinstance(tipo, str) else 'Sin clasificar'
    bar = '█' * min(count // 2, 30)
    print(f'  {label:<25} {count:>3}  {bar}')

print('\n── Desglose por modalidad ───────────────────────────────────────────────')
for mod, count in df_final['modalidad'].value_counts(dropna=False).items():
    label = mod if isinstance(mod, str) else 'Sin modalidad'
    print(f'  {label:<25} {count:>3}')

print('\n── Costos publicados ────────────────────────────────────────────────────')
print(f'  Con costo semestral COP:     {df_final["costo_semestral_cop"].notna().sum()}')
print(f'  Con costo semestral USD:     {df_final["costo_semestral_usd"].notna().sum()}')
print(f'  Con costo inscripción COP:   {df_final["costo_inscripcion_cop"].notna().sum()}')
sin_costo = (df_final['costo_semestral_cop'].isna() & df_final['costo_semestral_usd'].isna()).sum()
print(f'  Sin ningún costo publicado:  {sin_costo}')

print('\n── Acreditaciones ───────────────────────────────────────────────────────')
con_nac = ((df_final['acreditacion_nacional'].notna()) & (df_final['acreditacion_nacional'] != '')).sum()
con_int = ((df_final['acreditacion_internacional'].notna()) & (df_final['acreditacion_internacional'] != '')).sum()
print(f'  Con acreditación nacional:       {con_nac}')
print(f'  Con acreditación internacional:  {con_int}')

print('\n── SNIES y Título ───────────────────────────────────────────────────────')
con_snies  = ((df_final['codigo_snies'].notna()) & (df_final['codigo_snies'] != '')).sum()
con_titulo = ((df_final['titulo_otorgado'].notna()) & (df_final['titulo_otorgado'] != '')).sum()
print(f'  Con código SNIES:    {con_snies}')
print(f'  Con título otorgado: {con_titulo}')

print('\n── Facultad ─────────────────────────────────────────────────────────────')
con_fac = ((df_final['facultad'].notna()) & (df_final['facultad'] != '')).sum()
print(f'  Con facultad: {con_fac} / {len(df_final)}')
print('\n  Top 10 facultades:')
for fac, n in df_final['facultad'].value_counts().head(10).items():
    print(f'    {fac[:60]:<60}: {n}')

print('\n── Completitud de campos ────────────────────────────────────────────────')
for col in CSV_FIELDS:
    n   = ((df_final[col].notna()) & (df_final[col] != '')).sum()
    pct = n / len(df_final) * 100
    bar = '█' * int(pct / 5)
    print(f'  {col:<30} {n:>4}/{len(df_final)}  {pct:>5.1f}%  {bar}')

print('\n── Archivos generados ───────────────────────────────────────────────────')
for f_path in [CSV_FILE, MD_FILE, LOG_FILE]:
    if f_path.exists():
        size_kb = f_path.stat().st_size / 1024
        print(f'  ✅ {f_path.name:<40} {size_kb:.1f} KB')
    else:
        print(f'  ❌ {f_path.name} — no generado')

print('\n── URLs con error ───────────────────────────────────────────────────────')
if errores:
    for e in errores:
        print(f'  ❌ {e}')
else:
    print('  ✅ Sin errores')

print('\n' + '=' * 65)
print(' ¡Scraping completado!')
print('=' * 65)

 VALIDACIONES FINALES — Scraper Universidad de La Sabana

📊 Total de programas extraídos: 386

── Desglose por tipo ────────────────────────────────────────────────────
  Educación Continua        193  ██████████████████████████████
  Posgrado                  140  ██████████████████████████████
  Pregrado                   29  ██████████████
  Minor                      19  █████████
  Técnico Laboral             5  ██

── Desglose por modalidad ───────────────────────────────────────────────
  Virtual                   197
  Presencial                168
  Híbrido                    11
  Hyflex                     10

── Costos publicados ────────────────────────────────────────────────────
  Con costo semestral COP:     233
  Con costo semestral USD:     73
  Con costo inscripción COP:   134
  Sin ningún costo publicado:  123

── Acreditaciones ───────────────────────────────────────────────────────
  Con acreditación nacional:       17
  Con acreditación internacional:  143

── SNI

In [29]:
# ── Muestra de los registros más completos y de los con costo ─────────────────
df_work = df_final.copy()

print('🔍 Top 5 programas con más campos completos:\n')
df_work['_score'] = df_work[CSV_FIELDS].apply(
    lambda row: sum(1 for v in row if v and str(v).strip() not in ['', 'nan']), axis=1
)
cols_show = ['nombre','tipo','facultad','codigo_snies','titulo_otorgado',
             'costo_semestral_cop','acreditacion_nacional','_score']
display(df_work.nlargest(5, '_score')[cols_show])

print('\n🔍 Programas CON costo publicado (primeros 5):')
con_precio = df_work[df_work['costo_semestral_cop'].notna()][[
    'nombre', 'tipo', 'costo_semestral_cop', 'costo_semestral_usd', 'costo_inscripcion_cop'
]].head()
display(con_precio)

print('\n🔍 Programas CON acreditación nacional:')
con_acred = df_work[(df_work['acreditacion_nacional'].notna()) &
                    (df_work['acreditacion_nacional'] != '')][[
    'nombre', 'tipo', 'acreditacion_nacional'
]].head(10)
display(con_acred)

🔍 Top 5 programas con más campos completos:



,nombre,tipo,facultad,codigo_snies,titulo_otorgado,costo_semestral_cop,acreditacion_nacional,_score
1,Administración & Servicio,Pregrado,Escuela Internacional de Ciencias Económicas y...,104355.0,Profesional en Administración & Servicio,20321000.0,Alta Calidad mediante Resolución No. 016907 de...,16
2,Administración de Mercadeo y Logística Interna...,Pregrado,Escuela Internacional de Ciencias Económicas y...,101896.0,Administrador de Mercadeo y Logística Internac...,20321000.0,Alta Calidad mediante Resolución No. 024574 de...,16
200,Medicina,Pregrado,Relacionados Contacto Información del programa...,2518.0,Médico,34419000.0,Acreditación mediante Resolución No,16
4,Economía y Finanzas Internacionales,Pregrado,Facultad Continúa tu formación Pregrados Relac...,53516.0,Economista con énfasis en Finanzas Internacion...,20321000.0,Acreditación mediante Resolución No,15
5,Administración de Empresas,Pregrado,Escuela Internacional de Ciencias Económicas y...,1240.0,Administrador de Empresas,20321000.0,NaN,15



🔍 Programas CON costo publicado (primeros 5):


,nombre,tipo,costo_semestral_cop,costo_semestral_usd,costo_inscripcion_cop
1,Administración & Servicio,Pregrado,20321000.0,5110.0,130000.0
2,Administración de Mercadeo y Logística Interna...,Pregrado,20321000.0,5110.0,130000.0
3,Gastronomía,Pregrado,20321000.0,5110.0,130000.0
4,Economía y Finanzas Internacionales,Pregrado,20321000.0,5110.0,130000.0
5,Administración de Empresas,Pregrado,20321000.0,5110.0,130000.0



🔍 Programas CON acreditación nacional:


,nombre,tipo,acreditacion_nacional
1,Administración & Servicio,Pregrado,Alta Calidad mediante Resolución No. 016907 de...
2,Administración de Mercadeo y Logística Interna...,Pregrado,Alta Calidad mediante Resolución No. 024574 de...
4,Economía y Finanzas Internacionales,Pregrado,Acreditación mediante Resolución No
6,Administración de Negocios Internacionales,Pregrado,Acreditación mediante Resolución 017204 del 24...
24,Comunicación Social y Periodismo,Pregrado,Acreditación mediante Resolución 4000 del 18 d...
53,Derecho,Pregrado,Alta Calidad mediante Resolución No. 16769 del...
104,Maestría en Educación Bilingüe y Multicultural...,Posgrado,Alta Calidad según Resolución No. 002023 del 2...
108,Maestría en Educación,Posgrado,Acreditación mediante Resolución No
124,Maestría en Enfermería,Posgrado,alta calidad Recibió Acreditación de Alta Cali...
147,Filosofía,Pregrado,Alta Calidad: Resolución No. 016637 del 11 de ...
